# Module B · N1 — Audit FINAL-01Dataset `SIH26170-FINAL-01`. This notebook is the readable version of`scripts/01_audit.py` and `scripts/02_drift_structure.py`; both call the samefunctions in the `moduleb` package, so a number here and a number in `results/`cannot disagree.**What this notebook is for:** deciding whether anything in FINAL-01 is a *databug* to escalate to Chaitany, as opposed to a modelling problem Module B has tolive with. "The MAE is worse than I hoped" is never a data bug.

In [ ]:
import sys, pathlibROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()sys.path.insert(0, str(ROOT))import numpy as np, pandas as pdpd.set_option("display.width", 200); pd.set_option("display.max_columns", 80)from moduleb import (baselines, config, contract, cv, dataio, envelope,                     freeze, guards, metrics, models, reason_codes)from moduleb.constants import PARAMS, TARGET_COLSfrom moduleb.features import add_features, make_foldsprint("moduleb ready — frozen config digest", config.frozen_config_digest()[:16])

## 1. Load all three splits through the guards`load_split` runs every contract and quality check before it returns. If aforbidden column, a duplicate id or a non-positive measurement were present,this cell would raise rather than return a frame.

In [ ]:
splits = {n: dataio.load_split(n) for n in ("train", "calibration", "holdout")}base, limits = dataio.load_specs()for s in splits.values():    print(s.describe())print()print("dataset manifest:", dataio.load_dataset_version()["split_id"])

## 2. Whole-lot separationComponents in a lot are related. A single shared lot between two protectedsplits would turn a held-out score into a partly in-sample one.

In [ ]:
for a, b in (("train","calibration"), ("train","holdout"), ("calibration","holdout")):    guards.assert_lots_disjoint(splits[a].frame, splits[b].frame, name_a=a, name_b=b)    print(f"{a:12s} vs {b:12s}  disjoint")

## 3. Device_Specs coverageSeven of the eighteen variant × parameter cells carry no `static_spec_max`.Per decision D1 those stay empty — no limit is invented, and the limit-basedreason codes simply never fire there.

In [ ]:
cov = limits.notna().astype(int)display(cov)print("cells with no static limit:", int(limits.isna().to_numpy().sum()), "of 18")

## 4. Early vs late movement`r_early_late` measures the **linear** same-parameter early→late relationship. It is a diagnostic, not a ceiling — a low `r` does not rule out a nonlinear or multivariate route to the same target (corrected 19 Sep, review finding F6).A parameter with r² near zero is not a modelling failure — its late drift issimply not visible at 24 h, and saying so is part of the job.

In [ ]:
feat = add_features(splits["train"].frame, base)rows = []for v, g in feat.groupby("device_variant"):    for p in PARAMS:        x0 = g[f"{p}_0h"].to_numpy(float); x24 = g[f"{p}_24h"].to_numpy(float)        y  = g[f"{p}_168h"].to_numpy(float)        early, late = (x24 - x0) / x0, (y - x24) / x24        r = float(np.corrcoef(early, late)[0, 1])        rows.append(dict(variant=v, param=p, early_med_pct=100*np.median(early),                         late_med_pct=100*np.median(late), r_early_late=r, r2=r**2))display(pd.DataFrame(rows).round(4))

## 5. How much of the late drift is a LOT effect?Candidate V1 sat at 1.3–5.1%, which is why the review asked for stronger lotageing (accepted as generator change 1). This cell is the direct measurement ofwhether that change landed.

In [ ]:
rows = []for v, g in feat.groupby("device_variant"):    for p in PARAMS:        late = (g[f"{p}_168h"].to_numpy(float) - g[f"{p}_24h"].to_numpy(float)) / g[f"{p}_24h"].to_numpy(float)        s = pd.Series(late, index=g.lot_id.to_numpy())        m, n, gm = s.groupby(level=0).mean(), s.groupby(level=0).size(), s.mean()        share = float((n * (m - gm)**2).sum() / ((s - gm)**2).sum())        rows.append(dict(variant=v, param=p, lot_var_share_pct=100*share))display(pd.DataFrame(rows).round(3).pivot(index="param", columns="variant",                                          values="lot_var_share_pct").loc[PARAMS])

## 6. Cross-parameter drift structure, **within variant**Pooling A/B/C would manufacture correlation out of the three differentbaselines alone. This is the mechanism behind the timing group's`own+lot+cross` feature set.

In [ ]:
for v, g in feat.groupby("device_variant"):    late = pd.DataFrame({p: (g[f"{p}_168h"].to_numpy(float) - g[f"{p}_24h"].to_numpy(float))                            / g[f"{p}_24h"].to_numpy(float) for p in PARAMS})    print(v); display(late.corr().round(3))

## What to take awayRun `python scripts/01_audit.py` and `python scripts/02_drift_structure.py` forthe full tables, which are written to `results/`. Nothing in this notebookchanges any file.Next: **N2_benchmark.ipynb**.